In [ ]:
### https://medium.com/@subhraj07/building-intelligent-ai-agents-a-complete-architecture-guide-with-code-e5fbdd0755fc


In [1]:
from typing import List, Dict, Any
from datetime import datetime
import json
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

class MemorySystem:
    def __init__(self):
        # Short-term memory: Recent context (limited size)
        self.short_term_memory = []
        self.short_term_capacity = 10
        
        # Long-term memory: Persistent storage with embeddings
        self.long_term_memory = []
        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')
        
        # Initialize FAISS index for semantic search
        self.dimension = 384  # Embedding dimension
        self.index = faiss.IndexFlatL2(self.dimension)
        
    def add_to_short_term(self, item: Dict[str, Any]):
        """Add item to short-term memory with FIFO management"""
        item['timestamp'] = datetime.now().isoformat()
        self.short_term_memory.append(item)
        
        # Maintain capacity limit
        if len(self.short_term_memory) > self.short_term_capacity:
            # Move oldest item to long-term memory if important
            oldest = self.short_term_memory.pop(0)
            if self._is_important(oldest):
                self.add_to_long_term(oldest)
    
    def add_to_long_term(self, item: Dict[str, Any]):
        """Store item in long-term memory with embeddings"""
        # Create embedding for semantic search
        text = json.dumps(item)
        embedding = self.encoder.encode([text])[0]
        
        # Store in vector database
        self.index.add(np.array([embedding], dtype='float32'))
        self.long_term_memory.append(item)
    
    def _is_important(self, item: Dict[str, Any]) -> bool:
        """Determine if an item should be stored long-term"""
        # Implement importance scoring logic
        importance_keywords = ['error', 'success', 'learned', 'important']
        text = json.dumps(item).lower()
        return any(keyword in text for keyword in importance_keywords)
    
    def retrieve_relevant_memories(self, query: str, k: int = 5) -> List[Dict]:
        """Retrieve k most relevant memories for a query"""
        if not self.long_term_memory:
            return []
        
        # Encode query
        query_embedding = self.encoder.encode([query])[0]
        
        # Search in long-term memory
        distances, indices = self.index.search(
            np.array([query_embedding], dtype='float32'), k
        )
        
        relevant_memories = [self.long_term_memory[i] for i in indices[0] 
                           if i < len(self.long_term_memory)]
        
        # Combine with recent short-term memories
        recent_memories = self.short_term_memory[-3:] if self.short_term_memory else []
        
        return recent_memories + relevant_memories


/Users/sulbhajain/Documents/Personal/genAI_projects/langgraph_agents/lang_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from enum import Enum
from typing import Optional, Tuple

class AgentState(Enum):
    IDLE = "idle"
    THINKING = "thinking"
    PLANNING = "planning"
    EXECUTING = "executing"
    REFLECTING = "reflecting"
    
class Agent:
    def __init__(self):
        self.memory = MemorySystem()
        self.tools = ToolManager()
        self.planner = Planner()
        self.reflection = ReflectionModule()
        self.state = AgentState.IDLE
    # In Agent class, add:
    async def _process_step(self, step: 'Step') -> str:
        """Process a step without tools"""
        return f"Processed: {step.description}"
    
    def _should_replan(self, results: List) -> bool:
        """Check if replanning is needed"""
        return False
    
    def _format_results(self, results: List) -> str:
        """Format results for output"""
        return "\n".join(str(r) for r in results if r)
    async def process_request(self, user_input: str) -> str:
        """Main processing pipeline for user requests"""
        self.state = AgentState.THINKING
        
        # 1. Retrieve relevant context from memory
        context = self.memory.retrieve_relevant_memories(user_input)
        
        # 2. Generate a plan
        self.state = AgentState.PLANNING
        plan = await self.planner.create_plan(user_input, context)
        
        # 3. Execute the plan
        self.state = AgentState.EXECUTING
        result = await self._execute_plan(plan)
        
        # 4. Reflect on the execution
        self.state = AgentState.REFLECTING
        reflection = await self.reflection.analyze_execution(
            user_input, plan, result
        )
        
        # 5. Store experience in memory
        self.memory.add_to_short_term({
            'input': user_input,
            'plan': plan.to_dict(),
            'result': result,
            'reflection': reflection
        })
        
        self.state = AgentState.IDLE
        return result
    
    async def _execute_plan(self, plan: 'Plan') -> str:
        """Execute a plan step by step"""
        results = []
        
        for step in plan.steps:
            if step.requires_tool:
                # Use appropriate tool
                tool_result = await self.tools.execute(
                    step.tool_name, 
                    step.parameters
                )
                results.append(tool_result)
            else:
                # Process without tools
                result = await self._process_step(step)
                results.append(result)
                
            # Check if we should continue based on intermediate results
            if self._should_replan(results):
                new_plan = await self.planner.replan(plan, results)
                return await self._execute_plan(new_plan)
        
        return self._format_results(results)


In [ ]:
from dataclasses import dataclass
from typing import List, Optional

@dataclass
class Step:
    description: str
    requires_tool: bool
    tool_name: Optional[str]
    parameters: Optional[Dict]
    dependencies: List[int]  # Indices of steps this depends on

@dataclass
class Plan:
    goal: str
    steps: List[Step]
    
    def to_dict(self):
        return {
            'goal': self.goal,
            'steps': [vars(step) for step in self.steps]
        }

class Planner:
    def __init__(self):
        self.strategies = {
            'chain_of_thought': self._chain_of_thought_planning,
            'subgoal_decomposition': self._subgoal_decomposition,
            'self_critics': self._self_critics_planning
        }
    
    async def create_plan(self, goal: str, context: List[Dict]) -> Plan:
        """Create an execution plan for achieving the goal"""
        # Analyze the goal complexity
        complexity = self._assess_complexity(goal)
        
        # Choose planning strategy based on complexity
        if complexity < 3:
            strategy = 'chain_of_thought'
        elif complexity < 7:
            strategy = 'subgoal_decomposition'
        else:
            strategy = 'self_critics'
        
        # Generate plan using selected strategy
        plan = await self.strategies[strategy](goal, context)
        
        # Validate and optimize plan
        plan = self._optimize_plan(plan)
        
        return plan
    
    async def _chain_of_thought_planning(self, goal: str, context: List[Dict]) -> Plan:
        """Simple step-by-step planning"""
        steps = []
        
        # Simulate LLM call for generating steps
        thought_chain = await self._generate_thought_chain(goal)
        
        for i, thought in enumerate(thought_chain):
            step = Step(
                description=thought,
                requires_tool=self._needs_tool(thought),
                tool_name=self._identify_tool(thought) if self._needs_tool(thought) else None,
                parameters=self._extract_parameters(thought) if self._needs_tool(thought) else None,
                dependencies=list(range(i))  # Depends on all previous steps
            )
            steps.append(step)
        
        return Plan(goal=goal, steps=steps)
    
    async def _subgoal_decomposition(self, goal: str, context: List[Dict]) -> Plan:
        """Decompose goal into subgoals"""
        # Break down the main goal
        subgoals = await self._decompose_goal(goal)
        
        steps = []
        for subgoal in subgoals:
            # Create mini-plans for each subgoal
            subplan = await self._chain_of_thought_planning(subgoal, context)
            steps.extend(subplan.steps)
        
        return Plan(goal=goal, steps=steps)
    
    def _assess_complexity(self, goal: str) -> int:
        """Estimate task complexity (0-10 scale)"""
        # Simple heuristic based on goal characteristics
        complexity = 0
        
        # Check for multiple actions
        action_words = ['and', 'then', 'after', 'finally', 'also']
        complexity += sum(1 for word in action_words if word in goal.lower())
        
        # Check for complex domains
        complex_domains = ['analyze', 'research', 'optimize', 'design']
        complexity += sum(2 for domain in complex_domains if domain in goal.lower())
        
        return min(complexity, 10)
        
    async def _self_critics_planning(self, goal: str, context: List[Dict]) -> Plan:
        """Self-critical planning with multiple iterations"""
        # Start with initial plan
        plan = await self._chain_of_thought_planning(goal, context)
        
        # Self-critique and refine (simplified version)
        plan = await self._refine_plan(plan, goal)
        
        return plan
    
    async def _generate_thought_chain(self, goal: str) -> List[str]:
        """Generate step-by-step thoughts for achieving goal"""
        # Simulate LLM call to generate thought chain
        return [
            f"Understand the goal: {goal}",
            f"Break down into actionable steps",
            f"Identify required tools or resources",
            f"Execute steps sequentially"
        ]
    
    async def _decompose_goal(self, goal: str) -> List[str]:
        """Break down goal into subgoals"""
        # Simulate LLM call for goal decomposition
        return [goal]
    
    def _optimize_plan(self, plan: Plan) -> Plan:
        """Validate and optimize plan"""
        # Remove redundant steps
        unique_descriptions = []
        optimized_steps = []
        
        for step in plan.steps:
            if step.description not in unique_descriptions:
                unique_descriptions.append(step.description)
                optimized_steps.append(step)
        
        return Plan(goal=plan.goal, steps=optimized_steps)
    
    async def _refine_plan(self, plan: Plan, goal: str) -> Plan:
        """Refine plan through self-critique"""
        # Simplified refinement logic
        return plan
    
    def _needs_tool(self, thought: str) -> bool:
        """Check if a thought requires tool usage"""
        tool_keywords = ['search', 'calculate', 'fetch', 'analyze', 'execute', 'find']
        return any(keyword in thought.lower() for keyword in tool_keywords)
    
    def _identify_tool(self, thought: str) -> str:
        """Identify which tool is needed"""
        if 'search' in thought.lower():
            return 'search'
        elif 'calculate' in thought.lower():
            return 'calculator'
        elif 'calendar' in thought.lower():
            return 'calendar'
        return 'code_interpreter'
    
    def _extract_parameters(self, thought: str) -> Dict:
        """Extract parameters for tool execution"""
        return {'query': thought}
    
    async def replan(self, plan: Plan, results: List) -> Plan:
        """Replan based on intermediate results"""
        return plan


In [35]:
from abc import ABC, abstractmethod
import aiohttp
import subprocess
from datetime import datetime

class Tool(ABC):
    @abstractmethod
    async def execute(self, **kwargs) -> Dict[str, Any]:
        pass
    
    @abstractmethod
    def get_description(self) -> str:
        pass

class SearchTool(Tool):
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "https://api.search.example.com/v1"
    
    async def execute(self, query: str, **kwargs) -> Dict[str, Any]:
        """Execute web search"""
        # async with aiohttp.ClientSession() as session:
        #     headers = {"Authorization": f"Bearer {self.api_key}"}
        #     params = {"q": query, "limit": kwargs.get('limit', 10)}
            
        #     async with session.get(
        #         f"{self.base_url}/search", 
        #         headers=headers, 
        #         params=params
        #     ) as response:
        #         results = await response.json()
                
        #         return {
        #             'tool': 'search',
        #             'query': query,
        #             'results': results['items'],
        #             'timestamp': datetime.now().isoformat()
        #         }
        return {
            'tool': 'search',
            'query': query,
            'results': [
                {
                    'title': f'Result 1 for: {query}',
                    'url': 'https://example.com/1',
                    'snippet': f'Relevant information about {query}'
                },
                {
                    'title': f'Result 2 for: {query}',
                    'url': 'https://example.com/2',
                    'snippet': f'More details about {query}'
                }
            ],
            'timestamp': datetime.now().isoformat()
        }
    
    def get_description(self) -> str:
        return "Search the web for information"

class CodeInterpreterTool(Tool):
    def __init__(self, code: Optional[str] = None):
        self.supported_languages = ['python', 'javascript', 'bash']
        self.code = code
    
    async def execute(self, code: str = None, query: str = None, language: str = 'python', **kwargs) -> Dict[str, Any]:
        """Execute code safely in a sandboxed environment"""
        # Handle case where 'query' is passed instead of 'code'
        if code is None and query is not None:
            code = query
        
        if code is None:
            return {
                'tool': 'code_interpreter',
                'error': 'No code or query provided'
            }
        
        if language not in self.supported_languages:
            return {
                'tool': 'code_interpreter',
                'error': f'Unsupported language: {language}'
            }
        
        try:
            # Mock execution instead of actual subprocess
            return {
                'tool': 'code_interpreter',
                'language': language,
                'code': code,
                'output': f'Executed: {code}',
                'error': None,
                'execution_time': datetime.now().isoformat()
            }
        except Exception as e:
            return {
                'tool': 'code_interpreter',
                'error': str(e)
            }
    
        
        # try:
        #     # In production, use proper sandboxing (Docker, etc.)
        #     if language == 'python':
        #         result = subprocess.run(
        #             ['python', '-c', code],
        #             capture_output=True,
        #             text=True,
        #             timeout=30
        #         )
            
        #     return {
        #         'tool': 'code_interpreter',
        #         'language': language,
        #         'code': code,
        #         'output': result.stdout,
        #         'error': result.stderr if result.returncode != 0 else None,
        #         'execution_time': datetime.now().isoformat()
        #     }
        # except Exception as e:
        #     return {
        #         'tool': 'code_interpreter',
        #         'error': str(e)
        #     }
        # Handle case where 'query' is passed instead of 'code'
       
    def get_description(self) -> str:
        return "Execute code in Python, JavaScript, or Bash"

class CalendarTool(Tool):
    async def execute(self, action: str, **kwargs) -> Dict[str, Any]:
        """Manage calendar events"""
        if action == 'create':
            return await self._create_event(**kwargs)
        elif action == 'list':
            return await self._list_events(**kwargs)
        elif action == 'update':
            return await self._update_event(**kwargs)
        else:
            return {'error': f'Unknown action: {action}'}
    
    async def _create_event(self, title: str, date: str, **kwargs) -> Dict[str, Any]:
        # Implement calendar API integration
        return {
            'tool': 'calendar',
            'action': 'create',
            'event': {
                'title': title,
                'date': date,
                'id': 'evt_' + datetime.now().strftime('%Y%m%d%H%M%S')
            }
        }
    
    def get_description(self) -> str:
        return "Manage calendar events and schedules"
class CalculatorTool(Tool):
    async def execute(self, expression: str, **kwargs) -> Dict[str, Any]:
        """Evaluate mathematical expressions safely"""
        try:
            # Use a safe evaluation method
            import math
            allowed_names = {
                'abs': abs, 'round': round, 'min': min, 'max': max,
                'sum': sum, 'pow': pow, 'sqrt': math.sqrt, 'pi': math.pi,
                'e': math.e, 'sin': math.sin, 'cos': math.cos, 'tan': math.tan
            }
            
            result = eval(expression, {"__builtins__": {}}, allowed_names)
            
            return {
                'tool': 'calculator',
                'expression': expression,
                'result': result,
                'timestamp': datetime.now().isoformat()
            }
        except Exception as e:
            return {
                'tool': 'calculator',
                'expression': expression,
                'error': str(e)
            }
    
    def get_description(self) -> str:
        return "Calculate mathematical expressions"
        
class ToolManager:
    def __init__(self):
        self.tools = {
            'search': SearchTool(api_key='your_api_key'),
            'code_interpreter': CodeInterpreterTool(),
            'calendar': CalendarTool(),
            'calculator': CalculatorTool()
        }
    
    async def execute(self, tool_name: str, parameters: Dict) -> Dict[str, Any]:
        """Execute a tool with given parameters"""
        if tool_name not in self.tools:
            return {'error': f'Tool not found: {tool_name}'}
        
        tool = self.tools[tool_name]
        return await tool.execute(**parameters)
    
    def get_available_tools(self) -> List[str]:
        """Get list of available tools"""
        return list(self.tools.keys())
    
    def get_tool_descriptions(self) -> Dict[str, str]:
        """Get descriptions of all tools"""
        return {name: tool.get_description() 
                for name, tool in self.tools.items()}


In [28]:
class ReflectionModule:
    def __init__(self):
        self.performance_history = []
        self.learned_patterns = {}
    # In ReflectionModule class, add:
    def _extract_pattern(self, plan: Plan, result: str) -> str:
        """Extract successful pattern from execution"""
        return f"Pattern for: {plan.goal}"
    
    def _should_use_tool(self, description: str) -> bool:
        """Check if description suggests tool usage"""
        tool_keywords = ['search', 'calculate', 'analyze', 'fetch']
        return any(keyword in description.lower() for keyword in tool_keywords)
    
    def _dependencies_valid(self, step: 'Step', index: int, steps: List['Step']) -> bool:
        """Validate step dependencies"""
        return all(dep < index for dep in step.dependencies)
    
    def _has_similar_successful_pattern(self, plan: Plan) -> bool:
        """Check for similar successful patterns in history"""
        return len(self.performance_history) > 0
    
    def _generate_pattern_key(self, pattern: str) -> str:
        """Generate a key for pattern storage"""
        return hash(pattern) % 10000
    async def analyze_execution(self, 
                               input: str, 
                               plan: Plan, 
                               result: str) -> Dict[str, Any]:
        """Analyze execution and extract learnings"""
        reflection = {
            'input': input,
            'plan_quality': self._evaluate_plan_quality(plan),
            'execution_success': self._evaluate_result(result),
            'improvements': [],
            'learnings': []
        }
        
        # Identify what went well
        if reflection['execution_success'] > 0.7:
            reflection['learnings'].append({
                'type': 'successful_pattern',
                'pattern': self._extract_pattern(plan, result),
                'confidence': reflection['execution_success']
            })
        
        # Identify areas for improvement
        if reflection['plan_quality'] < 0.5:
            improvements = await self._generate_improvements(plan, result)
            reflection['improvements'] = improvements
        
        # Update learned patterns
        self._update_learned_patterns(reflection['learnings'])
        
        # Store in performance history
        self.performance_history.append(reflection)
        
        return reflection
    
    def _evaluate_plan_quality(self, plan: Plan) -> float:
        """Score plan quality from 0 to 1"""
        score = 1.0
        
        # Check for redundant steps
        descriptions = [step.description for step in plan.steps]
        unique_ratio = len(set(descriptions)) / len(descriptions)
        score *= unique_ratio
        
        # Check for proper tool usage
        for step in plan.steps:
            if self._should_use_tool(step.description) and not step.requires_tool:
                score *= 0.9
        
        # Check for logical dependencies
        for i, step in enumerate(plan.steps):
            if not self._dependencies_valid(step, i, plan.steps):
                score *= 0.8
        
        return max(0, min(1, score))
    
    def _evaluate_result(self, result: str) -> float:
        """Evaluate result quality"""
        # In practice, this would use more sophisticated metrics
        success_indicators = ['completed', 'success', 'done', 'achieved']
        failure_indicators = ['error', 'failed', 'unable', 'couldn\'t']
        
        result_lower = result.lower()
        
        success_score = sum(1 for indicator in success_indicators 
                          if indicator in result_lower)
        failure_score = sum(1 for indicator in failure_indicators 
                          if indicator in result_lower)
        
        if success_score + failure_score == 0:
            return 0.5  # Neutral
        
        return success_score / (success_score + failure_score)
    
    async def _generate_improvements(self, plan: Plan, result: str) -> List[str]:
        """Generate suggestions for improvement"""
        improvements = []
        
        # Analyze step efficiency
        if len(plan.steps) > 5:
            improvements.append("Consider consolidating steps for efficiency")
        
        # Check for missing error handling
        if 'error' in result.lower():
            improvements.append("Add error handling and recovery mechanisms")
        
        # Suggest alternative approaches
        if self._has_similar_successful_pattern(plan):
            improvements.append("Use previously successful pattern for similar tasks")
        
        return improvements
    
    def _update_learned_patterns(self, learnings: List[Dict]):
        """Update the knowledge base with new patterns"""
        for learning in learnings:
            pattern_key = self._generate_pattern_key(learning['pattern'])
            
            if pattern_key in self.learned_patterns:
                # Update existing pattern
                self.learned_patterns[pattern_key]['count'] += 1
                self.learned_patterns[pattern_key]['confidence'] = (
                    self.learned_patterns[pattern_key]['confidence'] * 0.9 + 
                    learning['confidence'] * 0.1
                )
            else:
                # Add new pattern
                self.learned_patterns[pattern_key] = {
                    'pattern': learning['pattern'],
                    'confidence': learning['confidence'],
                    'count': 1,
                    'first_seen': datetime.now().isoformat()
                }

In [34]:
import asyncio

async def main():
    # Initialize the agent
    agent = Agent()
    
    # Example: Research and summarize a topic
    user_request = "Research the latest developments in quantum computing and create a summary with the top 3 breakthroughs"
    
    print(f"User: {user_request}\n")
    print("Agent: Processing your request...\n")
    
    # The agent will:
    # 1. Check memory for related past research
    # 2. Create a plan (search → analyze → summarize)
    # 3. Execute tools (web search, content analysis)
    # 4. Reflect on the quality of results
    # 5. Store the experience for future use
    
    result = await agent.process_request(user_request)
    
    print(f"Agent: {result}\n")
    
    # Show the agent's reflection
    latest_memory = agent.memory.short_term_memory[-1]
    reflection = latest_memory['reflection']
    
    print("=" * 50)
    print("Agent's Self-Reflection:")
    print(f"Plan Quality: {reflection['plan_quality']:.2f}/1.00")
    print(f"Execution Success: {reflection['execution_success']:.2f}/1.00")
    
    if reflection['improvements']:
        print("\nAreas for Improvement:")
        for improvement in reflection['improvements']:
            print(f"  • {improvement}")
    
    if reflection['learnings']:
        print("\nWhat I Learned:")
        for learning in reflection['learnings']:
            print(f"  • {learning['type']}: {learning['confidence']:.2f} confidence")
    print ("=" * 50)
    print (reflection)
# Run the example
# if __name__ == "__main__":
#     asyncio.run(main())
await main()

User: Research the latest developments in quantum computing and create a summary with the top 3 breakthroughs

Agent: Processing your request...

Agent: {'tool': 'search', 'query': 'Understand the goal: Research the latest developments in quantum computing and create a summary with the top 3 breakthroughs', 'results': [{'title': 'Result 1 for: Understand the goal: Research the latest developments in quantum computing and create a summary with the top 3 breakthroughs', 'url': 'https://example.com/1', 'snippet': 'Relevant information about Understand the goal: Research the latest developments in quantum computing and create a summary with the top 3 breakthroughs'}, {'title': 'Result 2 for: Understand the goal: Research the latest developments in quantum computing and create a summary with the top 3 breakthroughs', 'url': 'https://example.com/2', 'snippet': 'More details about Understand the goal: Research the latest developments in quantum computing and create a summary with the top 3 br

In [33]:
reflection

NameError: name 'reflection' is not defined

In [30]:
async def _execute_parallel_steps(self, steps: List[Step]) -> List[Any]:
    """Execute independent steps in parallel"""
    tasks = []
    for step in steps:
        if not step.dependencies:  # No dependencies = can run in parallel
            task = self._execute_single_step(step)
            tasks.append(task)
    
    results = await asyncio.gather(*tasks)
    return results

In [31]:
class AdaptiveLearning:
    def __init__(self):
        self.success_patterns = {}
        self.failure_patterns = {}
    
    def learn_from_outcome(self, context: Dict, outcome: str):
        """Learn from successful and failed attempts"""
        pattern = self._extract_context_pattern(context)
        
        if self._is_success(outcome):
            self.success_patterns[pattern] = self.success_patterns.get(pattern, 0) + 1
        else:
            self.failure_patterns[pattern] = self.failure_patterns.get(pattern, 0) + 1
    
    def predict_success_probability(self, context: Dict) -> float:
        """Predict likelihood of success for given context"""
        pattern = self._extract_context_pattern(context)
        
        successes = self.success_patterns.get(pattern, 0)
        failures = self.failure_patterns.get(pattern, 0)
        
        if successes + failures == 0:
            return 0.5  # Unknown pattern
        
        return successes / (successes + failures)

In [32]:
class CostOptimizer:
    def __init__(self):
        self.cache = {}
        self.cost_per_action = {
            'llm_call': 0.002,
            'search': 0.001,
            'code_execution': 0.0001
        }
    
    def should_use_cache(self, query: str) -> bool:
        """Determine if cached result can be used"""
        if query in self.cache:
            cached_entry = self.cache[query]
            age = datetime.now() - cached_entry['timestamp']
            
            # Use cache if less than 1 hour old
            return age.total_seconds() < 3600
        return False
    
    def estimate_plan_cost(self, plan: Plan) -> float:
        """Estimate the cost of executing a plan"""
        total_cost = 0
        for step in plan.steps:
            if step.requires_tool:
                total_cost += self.cost_per_action.get(step.tool_name, 0.001)
            else:
                total_cost += self.cost_per_action['llm_call']
        return total_cost